# Independent Market-Making Bot on a Simulated Order Book

An educational notebook by Junhao Li. All input is synthetic; see README.md for methods, reproducibility, verification, and limitations. The original bilingual tutorial is retained below. Only the toy simulator is implemented: the exchange-integration checklist describes future work.

## 1. 先建立直觉：做市商在做什么？

把交易所想成一个二手市场。有人想立刻买，有人想立刻卖；**做市商（market maker）**同时给出愿意买入的价格和愿意卖出的价格，方便别人马上成交。

- **Bid（买价）**：我愿意买入的最高价格。
- **Ask / Offer（卖价）**：我愿意卖出的最低价格。
- **Spread（价差）**：`ask - bid`。若我低价买、高价卖，价差是潜在收入来源。
- **Order book（订单簿）**：所有公开买单和卖单的列表；最优买价叫 `best bid`，最优卖价叫 `best ask`。
- **Market order（市价单）**：不关心具体价格，只求立刻成交。
- **Limit order（限价单）**：只在不差于指定价格时成交；做市 bot 通常发限价单。

做市并不是无风险地“赚价差”：对手常在掌握信息时打你的单。例如你刚以 100 卖出，下一秒真实价值升到 101，这笔成交就是**逆向选择（adverse selection）**。

## 2. 术语表：

### 价格与订单簿

- **Mid price（中间价）**：`(best_bid + best_ask) / 2`。它是最简单的“当前合理价格”估计。
- **Tick size（最小报价单位）**：价格能变动的最小步长，例如 0.01。任何报价都要对齐 tick。
- **Depth（深度）**：某个价格上挂着多少数量；深度多不等于一定可信，因为订单可能被撤掉。
- **Microprice**：用买卖两边深度加权的 mid；当买方深度更大时，微价格会略高于 mid。它只是候选特征，不是真实价值。

### 仓位、收益与风险

- **Inventory / Position（库存/仓位）**：已经买入减去卖出的数量。`q > 0` 表示偏多，`q < 0` 表示偏空。
- **PnL（Profit and Loss，盈亏）**：策略赚或亏了多少。**Realized PnL** 来自已平仓交易；**Unrealized PnL** 是用当前标记价估计的浮动盈亏。
- **Markout**：成交后的一小段时间，市场价格朝对你有利还是不利方向移动；它用来诊断逆向选择。
- **Drawdown（回撤）**：净值从历史高点跌下来的幅度。高 PnL 但大回撤不一定是好策略。
- **Position limit（持仓限额）**：允许的最大绝对仓位。接近限额时必须减少同方向风险。

### 策略与工程

- **Fair value（公平价）**：bot 对“此刻合理价格”的估计，常从 mid 开始。
- **Reservation price（保留价）**：考虑库存后真正围绕它报价的中心。一个简单形式是 `r(q)=fair_value-k*q`；`k` 是库存惩罚强度。
- **Quote（报价）**：bot 给出的 bid 和 ask。
- **Baseline（基线策略）**：最简单、稳定、可测的版本；后续改动必须和它比较。
- **Replay（回放）**：把历史或模拟消息重新喂给策略，以便调试和比较。
- **A/B test（对照实验）**：除一个改动外保持完全相同，比较两版策略。

## 3. 真实交易所适配需要什么？（本 notebook 未实现）

下列清单是从教学模拟走向真实交易所适配时的后续工作，并非本 notebook 已实现的功能：

1. **入口程序**：平台能按要求启动它。
2. **市场数据处理器**：读入盘口/成交消息，维护最新状态。
3. **订单状态处理器**：记录自己的挂单、撤单、部分成交和订单回报。
4. **报价策略**：根据 fair value、库存和风险给出买卖价格及数量。
5. **风控模块**：持仓、消息、数据新鲜度或连接异常时限流、撤单或暂停。
6. **订单管理器**：避免重复下单、频繁 cancel/replace，以及撤单未确认时的误判。
7. **日志与可复现配置**：记录参数、订单、成交、仓位和 PnL，便于回放。
8. **提交包**：严格匹配官方 README 的文件结构、依赖和命令。

比赛包到手后的第一件事：把 README 中的产品、tick、撮合规则、订单 API、限额、评分/PnL 定义写成 checklist。


## 4. 导入库与模拟市场数据

真实比赛会通过官方 API 推送数据。为保证你在没有比赛包时也能练习，下面生成一个 toy market：价格随机游走、买卖价差固定、并给出两边深度。它**不能**代表真实市场，只用于验证程序结构。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dataclasses import dataclass, field

pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

## 1. 生成一个 toy market (simulated exchange tick data)

在正式比赛中，平台会给我们真实或官方模拟的订单簿数据：每一时刻的 best bid、best ask、挂单量、成交回报等。但在现在的练习阶段，我们还没有官方 API 和真实数据，所以需要自己写一个小型市场生成器

In [ ]:
def make_toy_market(n=2000, seed=7, start_price=100.0, spread=0.10):
    if not isinstance(n, (int, np.integer)) or n < 2 or spread <= 0:
        raise ValueError('n must be an integer >= 2 and spread must be positive')
    rng = np.random.default_rng(seed)
    imbalance = np.empty(n)
    imbalance[0] = rng.normal(0, 0.35)
    for i in range(1, n):
        imbalance[i] = 0.75 * imbalance[i - 1] + rng.normal(0, 0.28)
    imbalance = np.tanh(imbalance)  # 限制在 (-1, 1)

    mid = np.empty(n)
    mid[0] = start_price
    for i in range(1, n):
        # 不平衡对下一步收益有很弱信号，主要部分仍是噪声
        mid[i] = mid[i - 1] + 0.012 * imbalance[i - 1] + rng.normal(0, 0.025)

    total_depth = rng.integers(80, 160, n)
    bid_size = np.maximum(1, np.round(total_depth * (1 + imbalance) / 2)).astype(int)
    ask_size = np.maximum(1, total_depth - bid_size)

    return pd.DataFrame({
        't': np.arange(n),
        'best_bid': mid - spread / 2,
        'best_ask': mid + spread / 2,
        'bid_size': bid_size,
        'ask_size': ask_size,
        'true_imbalance': imbalance,  # 仅模拟器诊断用；策略不直接读取它 true_imbalance 是模拟器额外保存的内部答案，用于检查market数据是否按我们的设计生成。
# 策略本身不应该直接读取它。bot 只能看到公开可见的市场数据
    })

market = make_toy_market()
market.tail()


- n=2000：生成 2,000 个时间点。可以把每一个时间点理解为“交易所推送来的一次最新盘口快照”。它不一定对应真实的一秒；在高频交易中，一秒可以有很多次盘口更新。
- seed=7：随机数种子。市场里有随机波动。设置固定种子后，每次运行代码都会生成完全相同的 toy market。这样我们修改策略前后，面对的是同一份市场数据，结果才可以公平比较。
- start_price=100.0：初始中间价是 100。
- spread=0.10：市场买一价和卖一价之间的价差固定为 0.10。
- imbalance：用一个数字描述盘口买卖压力：imbalance > 0：买一档的挂单量更多，买方力量相对更强；imbalance 不能每一步都完全随机
  > 当前盘口压力，通常会受到上一时刻盘口压力的影响。当前买盘较强。下一时刻的 imbalance 不会突然完全忘记刚才的买盘压力，而是大致保留其中一部分：imbalance[i] = 0.75 * imbalance[i - 1] + rng.normal(0, 0.28) 下一刻买卖压力 = 上一刻买卖压力的延续 + 新的随机订单流
  > tanh(imbalance) 前面的随机过程可能产生非常大的数，例如 2.5 或 -3.0。但盘口不平衡需要有合理范围。我们希望-1 < imbalance < 1，np.tanh() 是一个把任意数字压缩到 (-1, 1) 的函数 （单调递增函数）
- 下一时刻价格 = 上一时刻价格 + 盘口压力带来的微弱影响 + 随机价格噪声 mid[i] = mid[i - 1] + 0.012 * imbalance[i - 1] + rng.normal(0, 0.025)
- total_depth ：个时间点随机生成一个 80 到 159 之间的整数。它表示最优买价和最优卖价两边合计的挂单量。

## 第一步：从订单簿得到 fair value

先实现两个估计器：mid 和 microprice。我们把它们封装成纯函数：输入一行市场状态，输出一个价格。纯函数便于单元测试和替换。

I would start with mid price as a robust baseline because it is simple and symmetric. Then I would test microprice as an additional short-horizon signal, since it incorporates top-of-book imbalance. However, I would not assume it is always superior: displayed depth can be cancelled quickly, so I would validate whether microprice improves post-trade markout and out-of-sample PnL before using it in live quoting.

- mid price：最中性的价格基线 base line。
- microprice：当 bid 深度较大时，更接近 ask；当 ask 深度较大时，更接近 bid。它是**短期特征**，不是真实价格。
- 对 bot 而言，**bid 的价格应向下取整**，ask 应向上取整。例如理论 bid=99.923，应报 99.92；理论 ask=99.923，应报 99.93。这样不会无意中让报价变得更激进。

In [ ]:
def mid_price(book):
    return (book['best_bid'] + book['best_ask']) / 2

def microprice(book):
    """
    Microprice（微观价格）：在 mid price 的基础上加入最优档挂单量。
    若 bid 档挂单量更大，通常表示短期买方力量相对更强， 价格可能更接近 ask，因此给 ask 更高权重 bid_size。
    若 ask 档挂单量更大，通常表示短期卖方力量相对更强，价格可能更接近 bid，因此给 bid 更高权重 aks_size。
    """
    bid, ask = book['best_bid'], book['best_ask']
    bid_sz, ask_sz = book['bid_size'], book['ask_size']
    total_depth = bid_sz + ask_sz
    return mid_price(book) if total_depth <= 0 else (ask * bid_sz + bid * ask_sz) / total_depth

def bid_to_tick(price, tick=0.01):
    '''买价向下对齐：不因取整而多付钱。'''
    return round(np.floor((price + 1e-12) / tick) * tick, 10)

def ask_to_tick(price, tick=0.01):
    '''卖价向上对齐：不因取整而少收钱。'''
    return round(np.ceil((price - 1e-12) / tick) * tick, 10)

In [ ]:
example = market.iloc[-1]
print(example)
print("-------------------------")
print('mid       :', round(mid_price(example), 4))
print('microprice:', round(microprice(example), 4))
print("--------------------------")
print('bid tick:99.923  :', bid_to_tick(99.923))
print('ask tick:99.923  :', ask_to_tick(99.923))

## 第二步 报价函数：库存偏多时整体下移，偏空时整体上移，把风控放在策略之前

风控不是在亏钱后才启用。每次准备报价前，先检查：

- 数据是否新鲜？
- 库存是否超过硬限额？
- 当前波动是否过高？
- 是否还有足够的消息预算？
> `quote_sizes` 分别返回 bid / ask 允许报价的数量；某侧为 0 表示该侧不报价。消息预算尚未实现。

- inventory：当前仓位。
- position_limit=20：最大允许持仓绝对值为 20。
- base_size=2：正常情况下每次报价 2 单位。
- stale=False：市场数据是否过期。
- volatile=False：市场是否处于高波动状态。

- 如果 stale=True：数据不新鲜，bot 不知道当前市场真实价格，不能用 return 0
- 如果仓位已经达到 +20 或 -20：关闭加仓侧，仍允许减仓侧。
- 买单容量为 position_limit - inventory；卖单容量为 position_limit + inventory。
- 每侧 size 分别按 base_size 和该侧容量的较小值截断。
- 高波动时进一步缩量: if volatile is False , 正常允许数量size, if True 报1个单位缩量
> 高波动意味着价格变化快。bot 的订单还挂在市场上时，fair value 可能已经变化；此时更容易发生逆向选择。所以高波动时通常会：
    - 加宽价差
    - 缩小每笔报价 size
    - 更积极撤掉旧单
    - 极端情况下暂停报价
    - 这段代码只实现了“缩量到 1”的最简单版本




- 最朴素的报价是 `bid=f-h`、`ask=f+h`，其中 `f` 是 fair value，`h` 是半价差。
- 若库存 `q>0`，说明已经偏多。我们不希望继续轻易买入，反而希望卖出一部分；因此把报价中心下调：`reservation = f - k*q`。注意：这不是预测市场会下跌，而是在控制自己的库存风险。

soft limit 和 half_spread=0.08 分别解决两个不同问题：

- soft limit：控制“仓位风险”，仓位不能超过 +20 或 -20，但如果 bot 到 +20 才开始限制买入，通常已经太晚了 （订单可能部分成交或延迟回报；你可能还有旧买单尚未撤）。
- half_spread：控制“报价离公平价有多远”，即成交机会、价差收入与逆向选择之间的平衡。报价价差 = 2 × half_spread，报价差至少要覆盖：+ 价格跳动风险 + 逆向选择风险 + 预期库存风险 + 手续费 + 期望利润（可以后续参数调优）
  > 报价更宽
            → 单笔潜在价差收入更大
            → 但成交机会通常更低
        
  > 报价更窄
            → 更容易成交
            → 但更容易被逆向选择，也更难覆盖费用和风险

When I am long inventory, I shift the reservation price downward. This lowers both quotes: the lower bid makes me less likely to buy more, while the lower ask makes my offer more competitive and helps me sell inventory. The half-spread is still applied around the shifted reservation price so that I continue to quote two-sided markets while controlling inventory risk.

In [ ]:
def quote_sizes(inventory, position_limit=20, soft_limit_ratio=0.75, base_size=2, stale=False, volatile=False):
    if any(not isinstance(v, (int, np.integer)) for v in (inventory, position_limit, base_size)):
        raise ValueError('inventory, position_limit, and base_size must be integers')
    if position_limit <= 0 or base_size < 0 or not 0 < soft_limit_ratio <= 1:
        raise ValueError('invalid position limit, size, or soft-limit ratio')
    if stale:
        return 0, 0
    if abs(inventory) > position_limit:
        raise RuntimeError('仓位已越过硬限额：真实系统应立即撤加仓单并告警')

    bid_size = min(base_size, position_limit - inventory)
    ask_size = min(base_size, position_limit + inventory)
    soft_limit = position_limit * soft_limit_ratio

    if inventory >= soft_limit:       # 偏多：只允许卖出减仓
        bid_size = 0
    elif inventory <= -soft_limit:    # 偏空：只允许买入减仓
        ask_size = 0

    if volatile:
        bid_size = min(bid_size, 1)
        ask_size = min(ask_size, 1)
    return bid_size, ask_size


def make_quote(fair_value, inventory, half_spread=0.08, inventory_k=0.025, tick=0.01):
    reservation = fair_value - inventory_k * inventory
    bid = bid_to_tick(reservation - half_spread, tick)
    ask = ask_to_tick(reservation + half_spread, tick)
    if bid >= ask:
        raise ValueError('报价交叉：请检查 tick 或 half_spread')
    return {'fair_value': fair_value, 'reservation': reservation, 'bid': bid, 'ask': ask}


In [ ]:
fair_value = 100.00

for q in [0, 14, 15, 18, 20, -18, -20]:
    bid_size, ask_size = quote_sizes(q)

    quote = make_quote(
        fair_value=fair_value,
        inventory=q,
        half_spread=0.08,
        inventory_k=0.025,
        tick=0.01,
    )

    # 即使 make_quote 计算了价格，size=0 的一侧实际上不会发送订单
    live_bid = quote["bid"] if bid_size > 0 else "不报价"
    live_ask = quote["ask"] if ask_size > 0 else "不报价"

    print(
        f"mid price ={fair_value} | "
        f"inventory={q} | "
        f"reservation={quote['reservation']:.2f} | "
        f"bid: size={bid_size}, price={live_bid} | "
        f"ask: size={ask_size}, price={live_ask}"
    )

## 第三步：建立 bot 状态：给机器人建立一份实时更新的“交易账本”。核心模块！！


机器人每秒会收到市场数据、自己订单的成交回报、撤单确认等消息。它必须随时知道：
- 我现在持有多少？
- 我花了多少钱、收了多少钱？
- 我还有哪些单挂在市场上？
- 刚才成交过什么？
- 按当前市场价格估值，我现在赚还是亏？

如果没有这份状态，bot 每次报价都像“失忆”一样，不知道自己已经买了多少、是否接近限仓，也无法正确控制风险。


In [ ]:
@dataclass
class BotState:
    inventory: int = 0
    cash: float = 0.0
    fills: list = field(default_factory=list)
    outstanding: dict = field(default_factory=dict)

    def inventory_bounds(self):
        # Diagnostic only: pending buys and sells must not offset worst-case exposure.
        buy_qty = sum(o['quantity'] for o in self.outstanding.values() if o['side'] == 'buy')
        sell_qty = sum(o['quantity'] for o in self.outstanding.values() if o['side'] == 'sell')
        return self.inventory - sell_qty, self.inventory + buy_qty

    def apply_fill(self, side, price, quantity, fee_per_unit=0.005):
        if side not in {'buy', 'sell'} or quantity <= 0:
            raise ValueError('side 必须为 buy/sell，且 quantity 必须为正数')
        signed_qty = quantity if side == 'buy' else -quantity
        self.inventory += signed_qty
        self.cash -= signed_qty * price
        self.cash -= fee_per_unit * quantity
        self.fills.append({'side': side, 'price': price, 'quantity': quantity, 'fee': fee_per_unit * quantity})

    def marked_pnl(self, mark_price):
        return self.cash + self.inventory * mark_price

In [ ]:
state = BotState()
state #即：刚开始没有仓位、没有现金收支、没有挂单、没有成交记录。

### 代码解读：
- BotState 是一个 Python 类。你可以把它理解为一个专门保存 bot 当前状态的对象。
- @dataclass 是 Python 提供的简化写法：它会自动帮你生成初始化函数 BotState() 即刚开始没有仓位、没有现金收支、没有挂单、没有成交记录。
- inventory 就是仓位，也可理解为“库存”，库存直接决定下一次如何报价，所以库存状态必须保存
- cash：交易后累计的现金流，记录 bot 因成交产生的现金变化
  > 买东西要付钱，所以现金减少;卖东西收到钱，所以现金增加。假设 bot 买入 2 单位资产，每单位价格为 100：cash = -200,
  > 假设之后 bot 卖出 1 单位，每单位价格为 101：cash = -200 + 101 = -99
  > 虽然账上现金仍是 -99，但 bot 还持有 1 单位资产，因此不能只看 cash 判断盈亏 --> marked_pnl
- outstanding：还挂在市场上、尚未完成的订单，bot 不能只看“已经成交的仓位”，还要看“可能很快成交的风险”
$$
[q_{min},q_{max}]=[q-\text{outstanding sell quantity},q+\text{outstanding buy quantity}]
$$
买卖挂单不能互相抵消最坏情形风险。当前模拟逐步成交，不使用 outstanding；inventory_bounds 仅为诊断示例，未接入异步订单管理器。

        如果： inventory = 18, position_limit = 20
        表面上看还可以再买 2 单位。但如果市场上还有两笔未成交的买单，每笔 2 单位，那么最坏情况下可能会再买 4 单位：18 + 4 = 22 超过限仓


- fills：已经成交的记录/成交历史
    > 为什么要保留成交历史，而不只保存最终仓位？因为你需要：
    - 排查某次仓位为什么突然变大；
    - 对账：仓位是否等于所有买入减去所有卖出；
    - 计算成交后 markout；
    - 分析哪些报价最容易被逆向选择；
    - 在比赛后解释策略表现, 比如你发现 PnL 下降，可以追问：
        - 是不是某个价格区间里，bot 连续在买单侧成交？
        - 是不是所有成交后价格都朝不利方向运动？
        - 是不是库存过大时没有及时缩量？

- apply_fill()：订单成交后，如何更新账本

- marked_pnl 是“按当前市场价格估值后的总盈亏”:
    - cash：已经收付的现金；
    - inventory * mark：当前持仓按最新市场价格值多少钱；
    - mark：通常取 mid price，或官方规则指定的标记价。

## 决策函数

这里只做三件事：先过风控；再估计 fair value；最后生成符合 tick 的报价。真实比赛中，返回值需要接到官方的下单、撤单和订单状态机。

In [ ]:
def decide(book, state, position_limit=20, base_size=2, stale=False, volatile=False):
    bid_size, ask_size = quote_sizes(
        state.inventory, position_limit=position_limit, base_size=base_size,
        stale=stale, volatile=volatile,
    )
    if bid_size == 0 and ask_size == 0:
        return {'action': 'cancel_all', 'reason': 'stale_data' if stale else 'zero_size'}

    # microprice 用于演示；真实项目必须和 mid baseline 做样本外 A/B test。
    fair = microprice(book)
    half_spread = 0.11 if volatile else 0.08
    quote = make_quote(fair, state.inventory, half_spread=half_spread)
    return {
        'action': 'quote', 'bid': quote['bid'], 'ask': quote['ask'],
        'bid_size': bid_size, 'ask_size': ask_size,
        'fair_value': fair, 'reservation': quote['reservation'],
    }

## 回测论证

这不是历史回测，也不模拟队列优先级。每步用依赖盘口不平衡的 Bernoulli 概率，直接在 bot 报价上全量成交。成交概率与报价距离、外部最优价和深度消耗无关；扩大价差不会降低成交概率，因此该机制会夸大价差收益，不能据此证明策略优势。

为了防止结果只靠一次随机种子，后面会运行多次种子并报告平均值与盈利比例。

In [ ]:
# 单次实验：seed=123 固定一次input
def run_toy_backtest(data, position_limit=20, base_size=2, fee_per_unit=0.005, seed=123):
    if len(data) < 2 or fee_per_unit < 0:
        raise ValueError('at least two snapshots and a nonnegative fee are required')
    rng = np.random.default_rng(seed)
    state, records = BotState(), []

    for i in range(len(data) - 1):
        book, next_book = data.iloc[i], data.iloc[i + 1]
        recent_move = 0.0 if i == 0 else mid_price(book) - mid_price(data.iloc[i - 1])
        volatile = abs(recent_move) > 0.05
        decision = decide(book, state, position_limit, base_size, volatile=volatile)
        buy_filled = sell_filled = 0

        if decision['action'] == 'quote':
            imbalance = (book['bid_size'] - book['ask_size']) / (book['bid_size'] + book['ask_size'])
            # 主动买方更可能在买盘强时出现，主动卖方则相反；有毒性但不是确定性。
            p_buy_taker = np.clip(0.075 + 0.030 * imbalance, 0.02, 0.13)
            p_sell_taker = np.clip(0.075 - 0.030 * imbalance, 0.02, 0.13)

            if decision['ask_size'] and rng.random() < p_buy_taker:
                state.apply_fill('sell', decision['ask'], decision['ask_size'], fee_per_unit)
                sell_filled = decision['ask_size']
            if decision['bid_size'] and rng.random() < p_sell_taker:
                state.apply_fill('buy', decision['bid'], decision['bid_size'], fee_per_unit)
                buy_filled = decision['bid_size']

        assert abs(state.inventory) <= position_limit, 'hard position limit breached'
        mark = mid_price(next_book)  # 成交后的下一时刻标记价，避免只看当前价
        records.append({
            't': book['t'], 'mid': mid_price(book), 'next_mid': mark,
            'inventory': state.inventory, 'pnl': state.marked_pnl(mark),
            'bid': decision.get('bid', np.nan), 'ask': decision.get('ask', np.nan),
            'buy_filled': buy_filled, 'sell_filled': sell_filled,
            'volatile': volatile,
        })

    result = pd.DataFrame(records)
    # Separate signed mid-price drift from fill-price markout; both are quantity-weighted.
    result['signed_fill'] = result['buy_filled'] - result['sell_filled']
    result['filled_quantity'] = result['buy_filled'] + result['sell_filled']
    result['signed_mid_move'] = result['signed_fill'] * (result['next_mid'] - result['mid'])
    result['fill_markout'] = (
        result['buy_filled'] * (result['next_mid'] - result['bid']).fillna(0)
        + result['sell_filled'] * (result['ask'] - result['next_mid']).fillna(0)
    )
    return result, state


result, final_state = run_toy_backtest(market, seed=123)
print('最终库存:', final_state.inventory)
print('最终 Marked PnL:', round(result['pnl'].iloc[-1], 2))
print('总买入 / 总卖出:', int(result['buy_filled'].sum()), '/', int(result['sell_filled'].sum()))
total_filled = result['filled_quantity'].sum()
print('每单位成交 1-step signed mid move:', round(result['signed_mid_move'].sum() / total_filled, 5) if total_filled else float('nan'))
print('每单位成交 1-step fill-price markout (before fees):', round(result['fill_markout'].sum() / total_filled, 5) if total_filled else float('nan'))

## 不挑种子：重复实验与风险诊断

下面固定同一条 seed=7 的合成市场路径，只改变 100 个订单流种子。该实验检查模拟机制的随机性与库存约束，不是 100 条独立市场路径，也不是样本外检验。盈利比例不能作为统计显著性、真实策略优势或比赛表现的证据。

In [ ]:
trial_pnls = []
trial_max_inventories = []
for seed in range(100):
    trial_result, _ = run_toy_backtest(market, seed=seed)
    trial_pnls.append(trial_result['pnl'].iloc[-1])
    trial_max_inventories.append(int(trial_result['inventory'].abs().max()))

print(f'100 次平均最终 PnL: {np.mean(trial_pnls):.2f}')
print(f'中位数最终 PnL: {np.median(trial_pnls):.2f}')
print(f'盈利运行比例: {(np.array(trial_pnls) > 0).mean():.1%}')
print(f'本次最大绝对库存: {result["inventory"].abs().max()}')
print(f'100 次最大绝对库存: {max(trial_max_inventories)}')

In [ ]:
running_max = result['pnl'].cummax().clip(lower=0.0)  # include initial zero equity
drawdown = result['pnl'] - running_max

fig, ax = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
ax[0].plot(result['t'], result['pnl'])
ax[0].axhline(0, color='black', linewidth=1)
ax[0].set_title('Marked PnL (marked at next mid)')

ax[1].plot(result['t'], result['inventory'])
ax[1].axhline(20, color='red', linestyle='--', label='hard limit')
ax[1].axhline(-20, color='red', linestyle='--')
ax[1].axhline(15, color='orange', linestyle=':', label='soft limit')
ax[1].axhline(-15, color='orange', linestyle=':')
ax[1].legend(); ax[1].set_title('Inventory')

ax[2].plot(result['t'], drawdown)
ax[2].axhline(0, color='black', linewidth=1)
ax[2].set_title('Drawdown')
ax[2].set_xlabel('time step')
plt.tight_layout()
plt.show()